# 1. Data Cleaning and Metrics Calculation
In this notebook, I load the raw data, clean it up, and calculate some key metrics.

In [ ]:
import pandas as pd
import numpy as np

### Load Data
First, let's read the raw CSV file.

In [ ]:
df = pd.read_csv('../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv')
df.head()

### Clean Data
I need to fix the column names, parse dates, and remove commas from numbers.

In [ ]:
df.columns = df.columns.str.replace('*', '', regex=False).str.strip()
df['Date'] = pd.to_datetime(df['Date'], format='%B %d, %Y', errors='coerce')
df = df.dropna(subset=['Date'])

for col in df.columns:
    if col != 'Date':
        df[col] = df[col].astype(str).str.replace(',', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

### Fill Missing Dates
There are some gaps in the dates. I will fill active counts by forward filling and set flow counts to 0.

In [ ]:
df = df.sort_values(by='Date').reset_index(drop=True)
date_range = pd.date_range(start=df['Date'].min(), end=df['Date'].max(), freq='D')
df = df.set_index('Date').reindex(date_range)
df.index.name = 'Date'

active_cols = ['Children in CBP custody', 'Children in HHS Care']
flow_cols = ['Children apprehended and placed in CBP custody', 'Children transferred out of CBP custody', 'Children discharged from HHS Care']

for col in active_cols:
    if col in df.columns:
        df[col] = df[col].ffill()

for col in flow_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df = df.fillna(0).reset_index()

### Calculate Metrics
Now I will calculate the metrics we need for the analysis.

In [ ]:
df['Total System Load'] = df['Children in CBP custody'] + df['Children in HHS Care']
df['Net Daily Intake'] = df['Children transferred out of CBP custody'] - df['Children discharged from HHS Care']

# Avoid division by zero
df['Care Load Growth Rate'] = np.where(df['Children in HHS Care'].shift(1) > 0,
                                       (df['Children in HHS Care'] - df['Children in HHS Care'].shift(1)) / df['Children in HHS Care'].shift(1) * 100, 0)

df['CBP Efficiency Ratio'] = np.where(df['Children in CBP custody'] > 0,
                                      df['Children transferred out of CBP custody'] / df['Children in CBP custody'] * 100, 0)

df['Discharge Capacity Ratio'] = np.where(df['Children in HHS Care'] > 0,
                                          df['Children discharged from HHS Care'] / df['Children in HHS Care'] * 100, 0)

df['Backlog Pressure Index'] = df['Net Daily Intake'].cumsum()

df['Net Daily Intake (7-day avg)'] = df['Net Daily Intake'].rolling(window=7, min_periods=1).mean()
df['System Load (7-day std)'] = df['Total System Load'].rolling(window=7, min_periods=1).std().fillna(0)
df['Discharge Capacity Ratio (14-day avg)'] = df['Discharge Capacity Ratio'].rolling(window=14, min_periods=1).mean()

df['Month'] = df['Date'].dt.to_period('M')
df['Week'] = df['Date'].dt.to_period('W')
df['Year'] = df['Date'].dt.year


### Save Processed Data

In [ ]:
df.to_csv('../data/processed/uac_metrics.csv', index=False)
print('Data successfully processed and saved!')